In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader,DirectoryLoader
from sklearn.manifold import TSNE
import plotly.graph_objects as go


c:\Users\Mohammed Baqar\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Mohammed Baqar\AppData\Local\Temp\ipykernel_20764\1790787501.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader,DirectoryLoader


In [2]:
MODEL = "gemini-2.5-flash"
db = "vector.db"
load_dotenv()
google_api_key = os.getenv("GEMINI_API_KEY")
GEMINI_BASE_URL="https://generativelanguage.googleapis.com/v1beta/openai/"
gemini=OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)


In [3]:
MODEL_1 = "gpt-4.5-nano"
google_api_key = os.getenv("GEMINI_API_KEY")


In [4]:
know = "knowledge-base/**/*.md"
files = glob.glob(know, recursive=True)
dic = []
for file in files:
    with open(file, 'r', encoding='utf-8') as f:
        dic+= f.read().splitlines()
        dic+="\n\n"
print(f"Total number of lines in the knowledge base: {len(dic)}")


Total number of lines in the knowledge base: 6186


In [5]:
text = "\n".join(dic)
en = tiktoken.encoding_for_model(MODEL_1)
tok = en.encode(text)
token_count = len(tok)
print(f"Total number of tokens in the knowledge base: {token_count}")


Total number of tokens in the knowledge base: 63568


In [6]:
folder = glob.glob('knowledge-base/*')
docs = []
for fold in folder:
    doc_type = os.path.basename(fold)
    loader = DirectoryLoader(fold, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    doc = loader.load()
    for d in doc:
        d.metadata["doc_type"] = doc_type
        docs.append(d)
print(f"Total number of documents in the knowledge base: {len(docs)}")


Total number of documents in the knowledge base: 76


In [ ]:
docs[1]


In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks = text_splitter.split_documents(docs)
print(f"Total number of documents after splitting: {len(chunks)}\n")
print(chunks[0].page_content)


Total number of documents after splitting: 393

# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.


In [13]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(db):
    vectordb = Chroma(persist_directory=db, embedding_function=embeddings).delete_collection()
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db)
print(f"Vector store created and persisted at: {db}")


Vector store created and persisted at: vector.db


In [14]:
collection = vectorstore._collection
count = collection.count()
print(f"Total number of documents in the vector store: {count}")


Total number of documents in the vector store: 393


In [17]:
sample_embedding = collection.get(limit=1,include=['embeddings'])['embeddings'][0]
dimensions = len(sample_embedding)
print(f"Dimension of the embeddings: {dimensions}")


Dimension of the embeddings: 384


In [ ]:
res = collection.get(include=['embeddings','metadatas','documents'])
vectors = np.array(res['embeddings'])
metadatas = res['metadatas']
documents = res['documents']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue','red','green','purple'][['company','products','contracts','employees'].index(t)]for t in doc_types]


In [15]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)
fig = go.Figure(
    data=[
        go.Scatter(
            x = reduced_vectors[:, 0],
            y = reduced_vectors[:, 1],
            mode = 'markers',
            marker=dict(size=8, color=colors, opacity=0.7),
            text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
            hoverinfo='text'
        )
    ]
)
fig.update_layout(
    title="2D vector visualization of document embeddings",
    scene = dict(xaxis_title='x', yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(l=0, r=0, b=0, t=30)
)
fig.show()


In [18]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)
fig = go.Figure(
    data=[
        go.Scatter3d(
            x = reduced_vectors[:, 0],
            y = reduced_vectors[:, 1],
            z = reduced_vectors[:, 2],
            mode = 'markers',
            marker=dict(size=8, color=colors, opacity=0.7),
            text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
            hoverinfo='text'
        )
    ]
)
fig.update_layout(
    title="3D vector visualization of document embeddings",
    scene = dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(l=10, r=10, b=10, t=40)
)
fig.show()
